# Chapter 34 – Worked examples

[Chapter 34](../index.md) has no numerical worked example: its subject is the plumbing between ToOp's stages.
This notebook runs that plumbing in miniature with `mini_pipeline.py`, sixty lines of standard library and
Pydantic that mirror `tests/fake_kafka.py`, `optimizer/ac/listener.py` and `optimizer/ac/storage.py`: a fake
broker, a result message, an in-memory SQLite store with a unique strategy hash, a poll function and a scrub.
The real thing, with ToOp's own classes, runs in [lab_toop.ipynb](lab_toop.ipynb) under ToOp's interpreter.

Run it from this folder (the kernel's working directory must be `src/`).

In [ ]:
import mini_pipeline

## A DC worker's results topic

The miniature DC run publishes one `started` message, five pushes (one topology repeated) and one `stopped`
message, serialized as JSON bytes, the way ToOp wraps Pydantic JSON in a Protobuf envelope.

In [ ]:
producer = mini_pipeline.dc_run()
msgs = producer.messages['results']
for raw in msgs:
    print(mini_pipeline.Result.model_validate_json(raw))

## The AC store: duplicates, the "added" dictionary and the stop message

`poll_results` stores every pushed topology; the unique hash rejects the repeated one with an `IntegrityError`
that is caught and ignored. When the batch also carries the `stopped` message, the count of that optimization is
deleted from the returned dictionary: the store has four rows, the caller is told about none of them. That is
Exercise 1.

In [ ]:
db = mini_pipeline.create_session()
print('with the stop message   :', mini_pipeline.poll_results(db, mini_pipeline.FakeConsumer({'results': msgs})))
print('rows stored             :', db.execute('SELECT optimization_id, actions, fitness FROM topology').fetchall())
db2 = mini_pipeline.create_session()
print('without the stop message:', mini_pipeline.poll_results(db2, mini_pipeline.FakeConsumer({'results': msgs[:-1]})))
print('the same messages again :', mini_pipeline.poll_results(db2, mini_pipeline.FakeConsumer({'results': msgs[:-1]})))

## State that lives only as long as the process

`scrub` deletes rows older than a day in ToOp; with `max_age_seconds=0` it empties the table, which is also what
a restart does to an in-memory database. A consumer with `kill_on_empty=True` raises once it has replayed
everything: that exception is how the integration test stops a worker loop that would otherwise run for ever.

In [ ]:
print('rows after scrub(0):', mini_pipeline.scrub(db, 0))
consumer = mini_pipeline.FakeConsumer({'results': msgs}, kill_on_empty=True)
print(len(consumer.consume()), 'messages consumed')
try:
    consumer.consume()
except mini_pipeline.FakeConsumerEmptyError as err:
    print('then:', type(err).__name__, '-', err)
db.close()
db2.close()